# Clustering and Split Visualization

Visualize:
- Cluster size distribution
- Train/val/test split composition
- Architecture representation across splits
- Verify no data leakage

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

from src.data.annotation_parser import AnnotationParser
from src.data.clustering import load_splits

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load annotations
annotation_csv = Path('../fragments_for_prediction_COREONLY.csv')
parser = AnnotationParser(annotation_csv)

# Load clusters
clusters_file = Path('../data/splits/clusters_70.json')
if clusters_file.exists():
    with open(clusters_file) as f:
        clusters = json.load(f)
    print(f'Loaded {len(clusters)} clusters')
else:
    print('Clusters file not found. Run 02_cluster_sequences.py first.')
    clusters = None

## Cluster Size Distribution

In [ ]:
if clusters:
    cluster_sizes = [len(members) for members in clusters.values()]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax = axes[0]
    ax.hist(cluster_sizes, bins=50, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Cluster Size')
    ax.set_ylabel('Count')
    ax.set_title('Cluster Size Distribution')
    ax.set_yscale('log')
    
    # Size categories
    ax = axes[1]
    size_cats = {
        'Singleton (1)': sum(1 for s in cluster_sizes if s == 1),
        'Small (2-5)': sum(1 for s in cluster_sizes if 2 <= s <= 5),
        'Medium (6-20)': sum(1 for s in cluster_sizes if 6 <= s <= 20),
        'Large (>20)': sum(1 for s in cluster_sizes if s > 20),
    }
    ax.bar(size_cats.keys(), size_cats.values(), color='steelblue')
    ax.set_xlabel('Cluster Size Category')
    ax.set_ylabel('Number of Clusters')
    ax.set_title('Cluster Size Categories')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Cluster size statistics:')
    print(f'  Min: {min(cluster_sizes)}')
    print(f'  Max: {max(cluster_sizes)}')
    print(f'  Mean: {np.mean(cluster_sizes):.1f}')
    print(f'  Median: {np.median(cluster_sizes)}')
    print(f'  Singletons: {sum(1 for s in cluster_sizes if s == 1)} ({sum(1 for s in cluster_sizes if s == 1)/len(cluster_sizes)*100:.1f}%)')

## Train/Val/Test Split Analysis

In [ ]:
# Load splits
splits_dir = Path('../data/splits')
if (splits_dir / 'train.txt').exists():
    train_ids, val_ids, test_ids = load_splits(splits_dir)
    print(f'Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}')
else:
    print('Splits not found. Run 03_create_splits.py first.')
    train_ids, val_ids, test_ids = set(), set(), set()

In [ ]:
if train_ids:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Split sizes
    ax = axes[0]
    sizes = [len(train_ids), len(val_ids), len(test_ids)]
    labels = ['Train', 'Val', 'Test']
    colors = ['#2ecc71', '#3498db', '#e74c3c']
    ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax.set_title('Split Distribution')
    
    # Architecture distribution per split
    ax = axes[1]
    
    def get_top_compositions(ids, n=5):
        comps = [parser.get(fid).fragment_composition for fid in ids if parser.get(fid)]
        return Counter(comps).most_common(n)
    
    # Get union of top compositions across all splits
    all_top = set()
    for ids in [train_ids, val_ids, test_ids]:
        for comp, _ in get_top_compositions(ids, 5):
            all_top.add(comp)
    
    # Count each composition per split
    x = np.arange(len(all_top))
    width = 0.25
    
    for i, (ids, label, color) in enumerate([
        (train_ids, 'Train', '#2ecc71'),
        (val_ids, 'Val', '#3498db'),
        (test_ids, 'Test', '#e74c3c')
    ]):
        comps = [parser.get(fid).fragment_composition for fid in ids if parser.get(fid)]
        counts = Counter(comps)
        values = [counts.get(comp, 0) for comp in all_top]
        ax.bar(x + i*width, values, width, label=label, color=color, alpha=0.8)
    
    ax.set_xlabel('Composition')
    ax.set_ylabel('Count')
    ax.set_title('Top Compositions by Split')
    ax.set_xticks(x + width)
    ax.set_xticklabels([c[:20] + '...' for c in all_top], rotation=45, ha='right', fontsize=8)
    ax.legend()
    
    plt.tight_layout()
    plt.show()

## Verify No Data Leakage

In [ ]:
if train_ids and val_ids and test_ids:
    # Check for overlap
    train_val_overlap = train_ids & val_ids
    train_test_overlap = train_ids & test_ids
    val_test_overlap = val_ids & test_ids
    
    print('Data Leakage Check:')
    print(f'  Train-Val overlap:  {len(train_val_overlap)} sequences')
    print(f'  Train-Test overlap: {len(train_test_overlap)} sequences')
    print(f'  Val-Test overlap:   {len(val_test_overlap)} sequences')
    
    if len(train_val_overlap) + len(train_test_overlap) + len(val_test_overlap) == 0:
        print('\n✓ No data leakage detected!')
    else:
        print('\n⚠ WARNING: Data leakage detected!')